# Mapping the Narrative Journey: A Structural Analysis of Podcast Conversations

**Notebook:** Initial analysis, testing, and implementation of semantic segmentation and topic modeling pipelines.

**Dataset:** SPoRC (Podcast Corpus)

**Goal:** Develop and evaluate methods to segment podcast transcripts into coherent narrative units.

In [ ]:
%pip install -r requirements.txt

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nltk
from nltk.tokenize import sent_tokenize
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.signal import find_peaks
nltk.download("punkt")
from datasets import load_dataset
import gc

# For topic modeling, graph modeling, visualization
from sklearn.cluster import KMeans
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
import networkx as nx
import plotly.graph_objects as go

## Load and Explore Data

#### Load the full SPoRC dataset

In [ ]:
# Load the SPoRC dataset (12G!)
#dataset = load_dataset("blitt/SPoRC", streaming=True)

#### Load sample subset of the SPoRC dataset

In [ ]:
# Path to your sample file
file_path = r"./episodeLevelDataSample.jsonl"
dataset_sample = pd.read_json(file_path, lines=True) # Reads JSONL line by line

In [ ]:
print(f"Number of episodes: {len(dataset_sample)}")
print(f"Columns: \n{list(dataset_sample.columns)}")
dataset_sample.head(2)

#### Preview first transcript

In [ ]:
episode = dataset_sample.iloc[0]["transcript"] # the first episodes transcript
print(f"Transcript sample: {episode[:500]}...") # preview the first 500 chars

### Semantic Segmentation Functions

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
def segment_fixed_threshold(text, model, threshold=0.7):
    sents = sent_tokenize(text)
    if len(sents) < 2: 
        return [text], []
    embs = model.encode(sents)
    sim = [cosine_similarity([embs[i]], [embs[i + 1]])[0][0] for i in range(len(embs) - 1)]
    breaks = [i + 1 for i, sc in enumerate(sim) if sc < threshold]
    idx = [0] + breaks + [len(sents)]
    segs = [" ".join(sents[idx[i]:idx[i + 1]]) for i in range(len(idx) - 1)]
    return segs, sim


def segment_adaptive(text, model, window=3, prominence=0.05, drop_factor=0.85):
    sents = sent_tokenize(text)
    if len(sents) < 2: 
        return [text], [], []
    embs = model.encode(sents)
    sim = [cosine_similarity([embs[i]], [embs[i + 1]])[0][0] for i in range(len(embs) - 1)]
    smooth = np.convolve(sim, np.ones(window)/window, mode="same")
    inverted = 1 - np.array(smooth)
    peaks, _ = find_peaks(inverted, prominence=prominence)
    breaks = [p for p in peaks if smooth[p] < drop_factor * np.mean(smooth)]
    idx = [0] + breaks + [len(sents)]
    segs = [" ".join(sents[idx[i]:idx[i + 1]]) for i in range(len(idx) - 1)]
    return segs, smooth, peaks


def segment_coherence(segments, model):
    vals = []
    for seg in segments:
        sents = sent_tokenize(seg)
        if len(sents) < 2: 
            continue
        embs = model.encode(sents)
        sims = [cosine_similarity([embs[i]], [embs[j]])[0][0]
                for i in range(len(embs)) for j in range(i + 1,len(embs))]
        vals.append(np.mean(sims))
    return np.mean(vals) if vals else 0

def plot_similarity_curve(sim_scores, peaks=None, title="Similarity Curve"):
    plt.figure(figsize=(8, 4))
    plt.plot(sim_scores, label="Semantic Similarity")

    if peaks is not None and len(peaks) > 0:
        plt.scatter(peaks, np.array(sim_scores)[peaks], color='red', label='Detected Boundaries')
    
    plt.title(title)
    plt.xlabel("Sentence index")
    plt.ylabel("Cosine similarity")
    plt.legend()
    plt.show()

### Test Semantic Segmentation on One Episode

In [ ]:
sample_text = dataset_sample.iloc[0]["transcript"]

segments_fixed, sim_fixed = segment_fixed_threshold(sample_text, model)
segments_adapt, sim_adapt, peaks = segment_adaptive(sample_text, model)

plot_similarity_curve(sim_fixed, title="Fixed Threshold Segmentation")
plot_similarity_curve(sim_adapt, peaks, title="Adaptive Segmentation")

print(f"Fixed: {len(segments_fixed)} segments \nAdaptive: {len(segments_adapt)} segments")
print(f"Fixed coherence: {segment_coherence(segments_fixed, model):.3f}")
print(f"Adaptive coherence: {segment_coherence(segments_adapt, model):.3f}")

### Evaluate on Multiple Episodes

In [ ]:
results = []
for i, row in dataset_sample.head(5).iterrows():
    text = row["transcript"]
    if not isinstance(text, str) or len(text.split()) < 50:
        continue

    seg_fixed, sim_fixed = segment_fixed_threshold(text, model)
    seg_adapt, sim_adapt, peaks = segment_adaptive(text, model)
    results.append({
        "episode_id": i,
        "epTitle": row["epTitle"],
        "podTitle": row["podTitle"],
        "num_sentences": len(sent_tokenize(text)),
        "fixed_segments": len(seg_fixed),
        "adaptive_segments": len(seg_adapt),
        "fixed_coherence": segment_coherence(seg_fixed, model),
        "adaptive_coherence": segment_coherence(seg_adapt, model),
        "overlapPropDuration": row.get("overlapPropDuration", np.nan),
        "numMainSpeakers": row.get("numMainSpeakers", np.nan),
        "avgTurnDuration": row.get("avgTurnDuration", np.nan)
    })
    
results_df = pd.DataFrame(results)
display(results_df)

results_df.plot.scatter(x="overlapPropDuration", y="adaptive_coherence", title="Overlap vs. Coherence")

### Prototype Topic Modeling and Graph Analysis

In [ ]:
n_topics = 10
vectorizer = CountVectorizer(ngram_range=(1, 2), stop_words="english")
kmeans = KMeans(n_clusters=n_topics, random_state=42)
topic_model = BERTopic(hdbscan_model=kmeans, vectorizer_model=vectorizer)

segments_dataset = pd.DataFrame({
    "episode_id": results_df["episode_id"].repeat(results_df["adaptive_segments"]).reset_index(drop=True),
    "text": [seg for ep in dataset_sample.head(5)["transcript"].apply(lambda x: segment_adaptive(x, model)[0]) for seg in ep]
})
topics, probs = topic_model.fit_transform(segments_dataset["text"])
segments_dataset["topic"] = topics

# Build topic graph
edges = []
for eid, group in segments_dataset.groupby("episode_id"):
    seq = list(group["topic"])
    for a,b in zip(seq[:-1], seq[1:]): edges.append((a,b))
G = nx.DiGraph()
for e in edges:
    if G.has_edge(e[0], e[1]):
        G[e[0]][e[1]]["weight"] += 1
    else:
        G.add_edge(e[0], e[1], weight=1)
    
plt.figure(figsize=(8, 6))
pos = nx.spring_layout(G, seed=42)
weights = [max(0.7, min(G[u][v]["weight"]/2, 3)) for u,v in G.edges()]
nx.draw(G, pos, with_labels=True, node_color="lightblue", node_size=400, width=weights, edgecolors="gray")
plt.title("Topic Transition Graph (Sample)", fontsize=14)
plt.show()

### Adaptive segmentation on 5 episodes

##### Load 5 episodes

In [ ]:
# Selecting first 5 episodes
num_episodes = 5
dataset_sample_subset = dataset_sample.head(num_episodes)

##### Adaptive Segmentation & Similarity Curves

In [ ]:
# Sentence transformer model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Adaptive segmentation and similarity curves
segments_list = []

for i, row in dataset_sample_subset.iterrows():
    transcript = row["transcript"]
    if not isinstance(transcript, str) or len(transcript.split()) < 50:
        continue
    
    # Apply adaptive segmentation
    segments, sim_scores, peaks = segment_adaptive(transcript, model)
    
    # Plot similarity curve for each episode
    plt.figure(figsize=(8,4))
    plt.plot(sim_scores, label="Semantic Similarity")
    if len(peaks) > 0:
        plt.scatter(peaks, np.array(sim_scores)[peaks], color='red', label='Detected Boundaries')
    plt.title(f"Episode {i} Similarity Curve")
    plt.xlabel("Sentence index")
    plt.ylabel("Cosine similarity")
    plt.legend()
    plt.show()
    
    # Save segments in list
    for seg in segments:
        segments_list.append({
            "episode_id": i,
            "epTitle": row.get("epTitle", f"Episode {i}"),
            "podTitle": row.get("podTitle", "Unknown Podcast"),
            "segment_text": seg
        })

# Convert to DataFrame
segments_df = pd.DataFrame(segments_list)
print(f"Total segments extracted: {len(segments_df)}")
segments_df.head()

##### Topic Modeling with BERTopic

In [ ]:
n_topics = 10
vectorizer = CountVectorizer(ngram_range=(1,2), stop_words="english")
kmeans = KMeans(n_clusters=n_topics, random_state=42)
topic_model = BERTopic(hdbscan_model=kmeans, vectorizer_model=vectorizer)

# Fit topic model
topics, probs = topic_model.fit_transform(segments_df["segment_text"])
segments_df["topic"] = topics

# Preview topics
topic_model.get_topic_info().head()

##### Topic Transition Graph

In [ ]:
edges = []
for eid, group in segments_df.groupby("episode_id"):
    seq = list(group["topic"])
    for a,b in zip(seq[:-1], seq[1:]):
        edges.append((a,b))

G = nx.DiGraph()
for e in edges:
    if G.has_edge(e[0], e[1]):
        G[e[0]][e[1]]["weight"] += 1
    else:
        G.add_edge(e[0], e[1], weight=1)

# Draw the graph
plt.figure(figsize=(8,6))
pos = nx.spring_layout(G, seed=42)
weights = [max(0.7, min(G[u][v]["weight"]/2, 3)) for u,v in G.edges()]
nx.draw(G, pos, with_labels=True, node_color="lightblue", node_size=400, width=weights, edgecolors="gray")
plt.title("Topic Transition Graph (Sample Episodes)", fontsize=14)
plt.show()

## Pipeline

### Semantic Segmentation

### Topic Labeling

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL = "meta-llama/Llama-3.2-1B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL)

In [ ]:
topic_labels = []
for topic_id in range(n_topics):
    document_samples = topic_model.get_representative_docs(topic_id)
    keywords = ", ".join([keyword for keyword, _ in topic_model.get_topic(topic_id)])
    excerpts = (f"Excerpt: {document_samples[0]}" if (len(document_samples) < 2)
                else "\n".join([f"Excerpt {i+1}: {excerpt}" for i, excerpt in enumerate(document_samples[:2])]))
    messages = [
        {"role": "user", "content": f"""
        OBJECTIVE 
        Given these keywords and excerpts from a podcast, make a title describing its content.
        Between THREE and FIVE words.
        Answer with just the title, nothing else. No comments or introduction.
        Only generate ONE title.
        Not punny.
        Always a couple of words long.
        Remember, never just one word.

        EXAMPLES START
        EXAMPLE 1
        Keywords: books, motif, reading, dark, theme
        Excerpt: This book really changed the way I look at the genre.
        Answer: Talking about Books

        EXAMPLE 2
        Keywords: walk, forest, mountain
        Excerpt: Going into nature and the forest relaxes me.
        Answer: Hiking for Health

        EXAMPLES OVER
         
        TASK START
        Keywords: {keywords}
        {excerpts}
        Answer:"""}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=10,
            temperature=0.1,
            do_sample=True,
            output_scores=True,
            return_dict_in_generate=True,
            pad_token_id=tokenizer.eos_token_id
        )
    
    generated_tokens = outputs.sequences[0][inputs['input_ids'].shape[1]:]
    response = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    cleaned_response = response.strip('"').replace("\n","")
    print(f"Topic {topic_id}\nKeywords: {keywords}\n{excerpts}\n{cleaned_response}\n")
    topic_labels.append(cleaned_response)

### Graph Modeling

### Visualization